In [2]:
# Adjust DU FH Timing Based on RU Logs - Prototype

import re
import yaml
from pathlib import Path
from IPython.display import display, Markdown

# === Step 1: Load DU Config and RU Log ===

du_conf_path = Path("sample_file/sample_en.conf")
ru_log_path = Path("sample_file/meta_ru_log/ap0_boot.log")

du_conf_text = du_conf_path.read_text(encoding="utf-8", errors="ignore")
ru_log_text = ru_log_path.read_text(encoding="utf-8", errors="ignore")

# Display preview
print("== Preview DU Config ==")
print("\n".join(du_conf_text.splitlines()[:20]))
print("\n== Preview RU Log ==")
print("\n".join(ru_log_text.splitlines()[:20]))


== Preview DU Config ==

###################################################################
# OAI gNodeB configuration file

Active_gNBs = ( "gNB-OAI");
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity = "none";

gNBs =
(
 {
    ////////// Identification parameters:
    gNB_ID    = 0xe01;             // gNB identifier (customizable), used to uniquely identify the gNB. This is 0xE01.
    gNB_name  = "gNB-OAI";         // Name of the gNB, for identification only with no protocol function.

    // Tracking area code, 0x0000 and 0xfffe are reserved values
    tracking_area_code  =  1;        // TAC (Tracking Area Code), used by the core network to identify the area (similar to LAC).

    plmn_list = ({ mcc = 001 ;mnc = 01 mnc_length = 2 snssaiList = ( { sst = 1 })})
    # Mobile Country Code (MCC), here set to 001 (for testing).

== Preview RU Log ==
Fri May 24 12:35:47 CST 2024
AP0 >> [CLI] Running.

>> Ethernet PHY Tx/Rx is stable
XPCS VR_MII_DIG_STS: 0x00000010
COP PORS

In [8]:
# === Step 2: Identify timing-related fields in DU config using LLM (simulated here) ===
timing_keywords = [
    "Tadv_cp_dl", "T2a_cp_dl", "T2a_cp_ul", "T2a_up", "Ta3",
    "T1a_cp_dl", "T1a_cp_ul", "T1a_up", "Ta4"
]
timing_fields_found = []
for line in du_conf_text.splitlines():
    for keyword in timing_keywords:
        if re.search(rf"\b{keyword}\b\s*=", line):
            timing_fields_found.append(line.strip())

print("\n[INFO] Identified timing-related config entries:")
for item in timing_fields_found:
    print(" -", item)


[INFO] Identified timing-related config entries:
 - Tadv_cp_dl = 125;    // Advance time for downlink control plane (unit: ns).
 - T2a_cp_dl = (259, 470);    // DL control plane DU -> RU max/min delay (ns).
 - T2a_cp_ul = (125, 1200);    // UL control plane RU -> DU max/min delay (ns).
 - T2a_up = (70, 345);    // UL user plane RU -> DU max/min delay (ns).
 - Ta3 = (50, 171);    // Time from DU receiving data to passing it to L1 (ns).
 - T1a_cp_dl = (258, 392);    // DL control plane DU processing time (DU to MAC packet send) in ns.
 - T1a_cp_ul = (285, 300);    // UL control plane DU processing time (ns).
 - T1a_up = (155, 300);    // UL user plane DU processing time (ns).
 - Ta4 = (0, 200);    // RU forwarding delay after receiving data (ns).


In [5]:

# === Step 3: Update DU Config timing section ===
updated_conf_text = re.sub(
    r"(sl_ahead\s*=\s*)\d+",
    rf"\1{suggested_sl_ahead}",
    du_conf_text
)

# Save new config
updated_conf_path = Path("/mnt/data/sample_en_updated.conf")
updated_conf_path.write_text(updated_conf_text, encoding="utf-8")

# === Step 4: Show diff preview ===
from difflib import unified_diff

diff = unified_diff(
    du_conf_text.splitlines(),
    updated_conf_text.splitlines(),
    fromfile="original_config.conf",
    tofile="updated_config.conf",
    lineterm=""
)

print("\n== Config Diff Preview ==")
for line in diff:
    print(line)

# Output path
print(f"\n✅ Updated config saved to: {updated_conf_path}")


error: invalid group reference 14 at position 1